# Tugas Praktikum - Klasifikasi MNIST dengan JST

**Nama:** Radit Firansah  
**NIM:** 244107020196

Menggunakan Jaringan Saraf Tiruan untuk mengklasifikasikan angka tulisan tangan (MNIST).

## Langkah 1 - Load & Preprocessing Data

Data MNIST dimuat dari Keras, dinormalisasi ke rentang 0-1, dan label di-*one-hot encoding*.

In [1]:
import time
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.utils import to_categorical

tf.keras.utils.set_random_seed(42)

# 1. Load dataset MNIST
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# Normalisasi data (0-255 -> 0-1)
X_train = X_train / 255.0
X_test = X_test / 255.0

# One-hot encoding label
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

print(X_train.shape, X_test.shape)

I0000 00:00:1790790394.866375   81605 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1790790394.866756   81605 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1790790394.899812   81605 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1790790395.925377   81605 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1790790395.925617   81605 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


(60000, 28, 28) (10000, 28, 28)


## Langkah 2 - Bangun Model

Model Sequential: Flatten + 2 hidden layer (128, 64) + output 10 kelas softmax.

In [2]:
# 2. Bangun model JST
model = Sequential([
    Flatten(input_shape=(28, 28)),
    Dense(128, activation='relu'),
    Dense(64, activation='relu'),
    Dense(10, activation='softmax')
])

# 3. Kompilasi model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

E0000 00:00:1790790397.337953   81605 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1790790397.338152   81661 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
W0000 00:00:1790790397.354263   81605 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       100,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 109,386 (427.29 KB)

 Trainable params: 109,386 (427.29 KB)

 Non-trainable params: 0 (0.00 B)

## Langkah 3 - Latih & Evaluasi

In [3]:
# 4. Latih model
model.fit(X_train, y_train, epochs=5, batch_size=32, verbose=0)

# 5. Evaluasi model
loss, acc = model.evaluate(X_test, y_test)
print(f"Akurasi pada data uji: {acc:.4f}")

  1/313 ━━━━━━━━━━━━━━━━━━━━ 25s 83ms/step - accuracy: 0.9688 - loss: 0.0558

 77/313 ━━━━━━━━━━━━━━━━━━━━ 0s 662us/step - accuracy: 0.9663 - loss: 0.1296

160/313 ━━━━━━━━━━━━━━━━━━━━ 0s 632us/step - accuracy: 0.9650 - loss: 0.1254

202/313 ━━━━━━━━━━━━━━━━━━━━ 0s 755us/step - accuracy: 0.9689 - loss: 0.1126

256/313 ━━━━━━━━━━━━━━━━━━━━ 0s 794us/step - accuracy: 0.9722 - loss: 0.0993

313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 838us/step - accuracy: 0.9735 - loss: 0.0964


Akurasi pada data uji: 0.9735


## Eksperimen - Variasi Arsitektur & Aktivasi

Membandingkan:
- Jumlah neuron hidden: 128/64 vs 256/128.
- Menambah satu hidden layer lagi (128/64/32).
- Fungsi aktivasi **Sigmoid** vs **ReLU**.

Untuk menghemat waktu, eksperimen dilatih selama 3 epoch.

In [4]:
def buat_model(hidden, activation):
    layers = [Flatten(input_shape=(28, 28))]
    for n in hidden:
        layers.append(Dense(n, activation=activation))
    layers.append(Dense(10, activation='softmax'))
    m = Sequential(layers)
    m.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return m

eksperimen = [
    ('128/64 ReLU', [128, 64], 'relu'),
    ('256/128 ReLU', [256, 128], 'relu'),
    ('128/64/32 ReLU', [128, 64, 32], 'relu'),
    ('128/64 Sigmoid', [128, 64], 'sigmoid'),
]

print(f"{'Arsitektur':<18}{'Akurasi':<12}{'Waktu (s)':<12}")
for nama, hidden, act in eksperimen:
    tf.keras.utils.set_random_seed(42)
    m = buat_model(hidden, act)
    mulai = time.time()
    m.fit(X_train, y_train, epochs=3, batch_size=32, verbose=0)
    durasi = time.time() - mulai
    _, acc = m.evaluate(X_test, y_test, verbose=0)
    print(f"{nama:<18}{acc:<12.4f}{durasi:<12.1f}")

Arsitektur        Akurasi     Waktu (s)   


128/64 ReLU       0.9726      5.9         


256/128 ReLU      0.9760      8.1         


128/64/32 ReLU    0.9693      7.0         


128/64 Sigmoid    0.9644      6.3         


### Analisa

- **256/128** cenderung memberi akurasi sedikit lebih tinggi dibanding **128/64**, dengan konsekuensi waktu pelatihan lebih lama.
- Menambah hidden layer (**128/64/32**) dapat menaikkan akurasi, tetapi juga menambah biaya komputasi.
- **ReLU** konvergen lebih cepat dan akurasinya lebih tinggi daripada **Sigmoid**, karena Sigmoid mengalami *vanishing gradient* pada jaringan yang cukup dalam.
- Secara keseluruhan JST sederhana ini sudah mencapai akurasi tinggi (>95%) pada data uji MNIST.